In [2]:
# ==========================================
# Import Required Libraries
# ==========================================

import pandas as pd
import numpy as np
import pickle

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

print("Libraries Imported Successfully")

Libraries Imported Successfully


In [3]:
# ==========================================
# Load CERT Datasets (Sample)
# ==========================================

NROWS = 100000

logon = pd.read_csv("/kaggle/input/datasets/andrihjonior/cert-insider-threat-dataset-r4-2/r4.2/logon.csv", nrows=NROWS)
device = pd.read_csv("/kaggle/input/datasets/andrihjonior/cert-insider-threat-dataset-r4-2/r4.2/device.csv", nrows=NROWS)
email = pd.read_csv("/kaggle/input/datasets/andrihjonior/cert-insider-threat-dataset-r4-2/r4.2/email.csv", nrows=NROWS)
file = pd.read_csv("/kaggle/input/datasets/andrihjonior/cert-insider-threat-dataset-r4-2/r4.2/file.csv", nrows=NROWS)
http = pd.read_csv("/kaggle/input/datasets/andrihjonior/cert-insider-threat-dataset-r4-2/r4.2/http.csv", nrows=NROWS)
insider = pd.read_csv("/kaggle/input/datasets/andrihjonior/cert-insider-threat-dataset-r4-2/answers/insiders.csv")

print("Datasets Loaded Successfully")

Datasets Loaded Successfully


In [4]:
datasets = {
    "Logon": logon,
    "Device": device,
    "Email": email,
    "File": file,
    "HTTP": http,
    "Insider": insider
}

for name, df in datasets.items():
    print(f"{name}: {df.shape}")

Logon: (100000, 5)
Device: (100000, 5)
Email: (100000, 11)
File: (100000, 6)
HTTP: (100000, 6)
Insider: (191, 6)


In [5]:
# Convert Date Columns

for df in [logon, device, email, file, http]:
    df["date"] = pd.to_datetime(df["date"])

print("Date conversion completed.")

Date conversion completed.


In [6]:
print("Logon Missing Values")
print(logon.isnull().sum())

print("\nDevice Missing Values")
print(device.isnull().sum())

print("\nEmail Missing Values")
print(email.isnull().sum())

print("\nFile Missing Values")
print(file.isnull().sum())

print("\nHTTP Missing Values")
print(http.isnull().sum())

Logon Missing Values
id          0
date        0
user        0
pc          0
activity    0
dtype: int64

Device Missing Values
id          0
date        0
user        0
pc          0
activity    0
dtype: int64

Email Missing Values
id                 0
date               0
user               0
pc                 0
to                 0
cc             56866
bcc            83036
from               0
size               0
attachments        0
content            0
dtype: int64

File Missing Values
id          0
date        0
user        0
pc          0
filename    0
content     0
dtype: int64

HTTP Missing Values
id         0
date       0
user       0
pc         0
url        0
content    0
dtype: int64


In [7]:
logon.drop_duplicates(inplace=True)
device.drop_duplicates(inplace=True)
email.drop_duplicates(inplace=True)
file.drop_duplicates(inplace=True)
http.drop_duplicates(inplace=True)

print("Duplicates Removed Successfully")

Duplicates Removed Successfully


In [8]:
print("Logon :", logon.shape)
print("Device :", device.shape)
print("Email :", email.shape)
print("File :", file.shape)
print("HTTP :", http.shape)
print("Insider :", insider.shape)

Logon : (100000, 5)
Device : (100000, 5)
Email : (100000, 11)
File : (100000, 6)
HTTP : (100000, 6)
Insider : (191, 6)


In [53]:
# ==========================================
# Logon Features
# ==========================================

logon["hour"] = logon["date"].dt.hour

logon["off_hours"] = (
    (logon["hour"] < 8) |
    (logon["hour"] > 18)
).astype(int)

logon["activity_date"] = logon["date"].dt.date

logon_features = logon.groupby(
    ["user", "activity_date"]
).agg(
    logon_count=("user", "count"),
    off_hours_logons=("off_hours", "sum"),
    distinct_pcs=("pc", "nunique")
).reset_index()

print(logon_features.head())
print("\nShape :", logon_features.shape)

      user activity_date  logon_count  off_hours_logons  distinct_pcs
0  AAE0190    2010-01-04            2                 0             1
1  AAE0190    2010-01-05            2                 0             1
2  AAE0190    2010-01-06            2                 0             1
3  AAE0190    2010-01-07            2                 0             1
4  AAE0190    2010-01-08            2                 0             1

Shape : (38735, 5)


In [54]:
# ==========================================
# Device (USB) Features
# ==========================================

device["hour"] = device["date"].dt.hour

device["off_hours"] = (
    (device["hour"] < 8) |
    (device["hour"] > 18)
).astype(int)

device["activity_date"] = device["date"].dt.date

device_features = device.groupby(
    ["user", "activity_date"]
).agg(
    usb_connects=("user", "count"),
    off_hours_usb=("off_hours", "sum")
).reset_index()

print(device_features.head())
print("\nShape :", device_features.shape)

      user activity_date  usb_connects  off_hours_usb
0  AAF0535    2010-01-05             4              0
1  AAF0535    2010-01-06             2              0
2  AAF0535    2010-01-07             4              0
3  AAF0535    2010-01-08             8              0
4  AAF0535    2010-01-11             2              0

Shape : (14076, 4)


In [55]:
# ==========================================
# File Features
# ==========================================

file["activity_date"] = file["date"].dt.date

file_features = file.groupby(
    ["user", "activity_date"]
).agg(
    files_copied_to_usb=("user", "count")
).reset_index()

file_features["sensitive_files_to_usb"] = 0

print("File Features")
print(file_features.head())
print("Shape :", file_features.shape)


# ==========================================
# Email Features
# ==========================================

email["external"] = (
    email["to"].astype(str).str.contains("@", na=False)
).astype(int)

email["activity_date"] = email["date"].dt.date

email_features = email.groupby(
    ["user", "activity_date"]
).agg(
    total_emails_sent=("user", "count"),
    external_emails_sent=("external", "sum"),
    total_attachments=("attachments", "sum"),
    total_email_size=("size", "sum")
).reset_index()

print("\nEmail Features")
print(email_features.head())
print("Shape :", email_features.shape)


# ==========================================
# HTTP Features
# ==========================================

http["cloud"] = http["url"].str.contains(
    "dropbox|drive|mega|onedrive|box",
    case=False,
    na=False
).astype(int)

http["activity_date"] = http["date"].dt.date

http_features = http.groupby(
    ["user", "activity_date"]
).agg(
    http_requests=("user", "count"),
    cloud_job_visits=("cloud", "sum")
).reset_index()

print("\nHTTP Features")
print(http_features.head())
print("Shape :", http_features.shape)

File Features
      user activity_date  files_copied_to_usb  sensitive_files_to_usb
0  AAF0535    2010-01-05                    1                       0
1  AAF0535    2010-01-06                    5                       0
2  AAF0535    2010-01-07                    1                       0
3  AAF0535    2010-01-08                    3                       0
4  AAF0535    2010-01-12                    2                       0
Shape : (10487, 4)

Email Features
      user activity_date  total_emails_sent  external_emails_sent  \
0  AAE0190    2010-01-04                 14                    14   
1  AAE0190    2010-01-05                 13                    13   
2  AAE0190    2010-01-06                 14                    14   
3  AAE0190    2010-01-07                 14                    14   
4  AAE0190    2010-01-08                 13                    13   

   total_attachments  total_email_size  
0                  4            441328  
1                  2            35

In [56]:
# ==========================================
# Merge All Features
# ==========================================

merge_keys = ["user", "activity_date"]

daily_user_features = logon_features.merge(
    device_features,
    on=merge_keys,
    how="outer"
)

daily_user_features = daily_user_features.merge(
    file_features,
    on=merge_keys,
    how="outer"
)

daily_user_features = daily_user_features.merge(
    email_features,
    on=merge_keys,
    how="outer"
)

daily_user_features = daily_user_features.merge(
    http_features,
    on=merge_keys,
    how="outer"
)

daily_user_features.fillna(0, inplace=True)

print("=" * 60)
print("MERGED FEATURE DATASET")
print("=" * 60)

print("\nShape :", daily_user_features.shape)
print("Unique Users :", daily_user_features["user"].nunique())
print("Unique Dates :", daily_user_features["activity_date"].nunique())

daily_user_features.head()

MERGED FEATURE DATASET

Shape : (45995, 15)
Unique Users : 1000
Unique Dates : 115


,user,activity_date,logon_count,off_hours_logons,distinct_pcs,usb_connects,off_hours_usb,files_copied_to_usb,sensitive_files_to_usb,total_emails_sent,external_emails_sent,total_attachments,total_email_size,http_requests,cloud_job_visits
0,AAE0190,2010-01-04,2.0,0.0,1.0,0.0,0.0,0.0,0.0,14.0,14.0,4.0,441328.0,143.0,4.0
1,AAE0190,2010-01-05,2.0,0.0,1.0,0.0,0.0,0.0,0.0,13.0,13.0,2.0,355552.0,0.0,0.0
2,AAE0190,2010-01-06,2.0,0.0,1.0,0.0,0.0,0.0,0.0,14.0,14.0,12.0,532647.0,0.0,0.0
3,AAE0190,2010-01-07,2.0,0.0,1.0,0.0,0.0,0.0,0.0,14.0,14.0,9.0,474631.0,0.0,0.0
4,AAE0190,2010-01-08,2.0,0.0,1.0,0.0,0.0,0.0,0.0,13.0,13.0,10.0,400637.0,0.0,0.0


In [15]:
print("Training Dataset Shape:")
print(daily_user_features.shape)

Training Dataset Shape:
(1000, 14)


In [16]:
daily_user_features.describe()

,logon_count,off_hours_logons,distinct_pcs,usb_connects,off_hours_usb,files_copied_to_usb,sensitive_files_to_usb,total_emails_sent,external_emails_sent,total_attachments,total_email_size,http_requests,cloud_job_visits
count,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.0,1000.000000,1000.000000,1000.000000,1.000000e+03,1000.000000,1000.000000
mean,100.000000,30.832000,5.780000,100.000000,6.889000,100.000000,0.0,100.000000,100.000000,45.743000,2.998839e+06,100.000000,1.864000
std,50.024899,39.550095,21.244079,295.672563,42.885824,317.870697,0.0,65.509889,65.509889,46.788504,1.968932e+06,75.446485,4.049927
min,58.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.0,9.000000,9.000000,0.000000,2.477180e+05,9.000000,0.000000
25%,75.000000,5.750000,1.000000,0.000000,0.000000,0.000000,0.0,36.000000,36.000000,4.000000,1.057231e+06,29.000000,0.000000
50%,75.000000,25.000000,1.000000,0.000000,0.000000,0.000000,0.0,107.000000,107.000000,31.000000,3.223617e+06,99.000000,0.000000
75%,106.000000,38.000000,2.000000,0.000000,0.000000,0.000000,0.0,138.000000,138.000000,76.250000,4.136571e+06,143.000000,1.000000
max,405.000000,265.000000,145.000000,2318.000000,493.000000,2400.000000,0.0,442.000000,442.000000,305.000000,1.352244e+07,503.000000,37.000000


In [17]:
daily_user_features.to_csv(
    "daily_user_features.csv",
    index=False
)

print("daily_user_features.csv saved successfully.")

daily_user_features.csv saved successfully.


In [29]:
# ==========================================
# Heuristic Label Generation
# ==========================================

feature_columns = [
    "files_copied_to_usb",
    "external_emails_sent",
    "off_hours_logons",
    "distinct_pcs",
    "sensitive_files_to_usb",
    "cloud_job_visits",
    "off_hours_usb"
]

feature_columns = [
    col for col in feature_columns
    if col in daily_user_features.columns
]

means = daily_user_features[feature_columns].mean()


def assign_label(row):

    if (
        row.get("files_copied_to_usb", 0)
        > means.get("files_copied_to_usb", 1) * 1.5
        or
        row.get("external_emails_sent", 0)
        > means.get("external_emails_sent", 1) * 1.5
    ):
        return 0   # Data Exfiltration

    elif (
        row.get("off_hours_logons", 0)
        > means.get("off_hours_logons", 1) * 1.5
        or
        row.get("distinct_pcs", 0)
        > means.get("distinct_pcs", 1) * 1.5
    ):
        return 1   # IT Sabotage

    elif (
        row.get("sensitive_files_to_usb", 0)
        > means.get("sensitive_files_to_usb", 1) * 1.5
        or
        row.get("cloud_job_visits", 0)
        > means.get("cloud_job_visits", 1) * 1.5
    ):
        return 2   # Intellectual Property Theft

    elif (
        row.get("off_hours_usb", 0)
        > means.get("off_hours_usb", 1) * 1.5
    ):
        return 4   # Unauthorized Access

    else:
        return 3   # Normal


daily_user_features["Label"] = daily_user_features.apply(
    assign_label,
    axis=1
)

print("Label Distribution:")
print(daily_user_features["Label"].value_counts())

Label Distribution:
Label
3    497
0    333
2    111
1     51
4      8
Name: count, dtype: int64


In [31]:
# ==========================================
# Prepare Training Features
# ==========================================

drop_cols = ["user", "day", "date", "Label"]

X = daily_user_features.drop(
    columns=[c for c in drop_cols if c in daily_user_features.columns],
    errors="ignore"
)

# Keep only numeric columns
X = X.select_dtypes(include=["number"])

# Replace missing values
X = X.fillna(0)

# Target
y = daily_user_features["Label"]

# Save exact feature order
feature_columns = list(X.columns)

print("Feature Columns:")
print(feature_columns)

print("\nFeature Shape:", X.shape)
print("Target Shape:", y.shape)

Feature Columns:
['logon_count', 'off_hours_logons', 'distinct_pcs', 'usb_connects', 'off_hours_usb', 'files_copied_to_usb', 'sensitive_files_to_usb', 'total_emails_sent', 'external_emails_sent', 'total_attachments', 'total_email_size', 'http_requests', 'cloud_job_visits']

Feature Shape: (1000, 13)
Target Shape: (1000,)


In [32]:
# ==========================================
# Scaling and Train-Test Split
# ==========================================

from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split

scaler = MinMaxScaler()

X_scaled = scaler.fit_transform(X)

X_scaled = pd.DataFrame(
    X_scaled,
    columns=X.columns
)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled,
    y,
    test_size=0.3,
    random_state=42
)

print("X_train :", X_train.shape)
print("X_test  :", X_test.shape)
print("y_train :", y_train.shape)
print("y_test  :", y_test.shape)

X_train : (700, 13)
X_test  : (300, 13)
y_train : (700,)
y_test  : (300,)


In [33]:
# ==========================================
# Gradient Boosting Model Training
# ==========================================

from sklearn.ensemble import GradientBoostingClassifier

gb = GradientBoostingClassifier(
    n_estimators=100,
    random_state=42
)

gb.fit(X_train, y_train)

print("Gradient Boosting Model Trained Successfully")

Gradient Boosting Model Trained Successfully


In [57]:
# ==========================================
# Behavioral Baseline
# ==========================================

baseline_features = [
    "logon_count",
    "off_hours_logons",
    "distinct_pcs",
    "usb_connects",
    "off_hours_usb",
    "files_copied_to_usb",
    "sensitive_files_to_usb",
    "total_emails_sent",
    "external_emails_sent",
    "total_attachments",
    "total_email_size",
    "http_requests",
    "cloud_job_visits"
]

behavior_baseline = daily_user_features.groupby("user")[
    baseline_features
].mean()

print("=" * 60)
print("BEHAVIORAL BASELINE")
print("=" * 60)

print("\nUsers :", behavior_baseline.shape[0])
print("Features :", behavior_baseline.shape[1])

behavior_baseline.head()

BEHAVIORAL BASELINE

Users : 1000
Features : 13


,logon_count,off_hours_logons,distinct_pcs,usb_connects,off_hours_usb,files_copied_to_usb,sensitive_files_to_usb,total_emails_sent,external_emails_sent,total_attachments,total_email_size,http_requests,cloud_job_visits
user,,,,,,,,,,,,,
AAE0190,1.973684,0.000000,1.000000,0.00000,0.0,0.000000,0.0,4.368421,4.368421,1.631579,132731.763158,3.763158,0.105263
AAF0535,0.961538,0.000000,0.487179,3.74359,0.0,1.794872,0.0,0.474359,0.474359,0.538462,14123.807692,0.371795,0.000000
AAF0791,1.973684,0.000000,1.000000,0.00000,0.0,0.000000,0.0,2.815789,2.815789,0.000000,85967.526316,2.552632,0.052632
AAL0706,1.973684,1.000000,1.000000,0.00000,0.0,0.000000,0.0,0.315789,0.315789,0.078947,8645.763158,0.263158,0.000000
AAM0658,1.973684,0.973684,1.000000,0.00000,0.0,0.000000,0.0,0.947368,0.947368,0.763158,30004.184211,0.763158,0.000000


In [58]:
# ==========================================
# Behavioral Deviation
# ==========================================

deviation_data = daily_user_features.merge(
    behavior_baseline.reset_index(),
    on="user",
    suffixes=("", "_baseline")
)

for feature in baseline_features:
    deviation_data[f"{feature}_deviation"] = (
        deviation_data[feature]
        - deviation_data[f"{feature}_baseline"]
    )

deviation_columns = [
    feature + "_deviation"
    for feature in baseline_features
]

print("=" * 60)
print("BEHAVIOR DEVIATION ANALYSIS")
print("=" * 60)

print("\nShape :", deviation_data.shape)
print("Deviation Features :", len(deviation_columns))

print("\nSample:")
print(
    deviation_data[
        [
            "user",
            "activity_date",
            "logon_count",
            "logon_count_baseline",
            "logon_count_deviation"
        ]
    ].head(10)
)

BEHAVIOR DEVIATION ANALYSIS

Shape : (45995, 41)
Deviation Features : 13

Sample:
      user activity_date  logon_count  logon_count_baseline  \
0  AAE0190    2010-01-04          2.0              1.973684   
1  AAE0190    2010-01-05          2.0              1.973684   
2  AAE0190    2010-01-06          2.0              1.973684   
3  AAE0190    2010-01-07          2.0              1.973684   
4  AAE0190    2010-01-08          2.0              1.973684   
5  AAE0190    2010-01-11          2.0              1.973684   
6  AAE0190    2010-01-12          2.0              1.973684   
7  AAE0190    2010-01-13          2.0              1.973684   
8  AAE0190    2010-01-14          2.0              1.973684   
9  AAE0190    2010-01-15          2.0              1.973684   

   logon_count_deviation  
0               0.026316  
1               0.026316  
2               0.026316  
3               0.026316  
4               0.026316  
5               0.026316  
6               0.026316  
7       

In [60]:
# ==========================================
# Save Anomaly Results
# ==========================================

import os
import joblib

os.makedirs("dataset", exist_ok=True)

deviation_data.to_csv(
    "dataset/anomaly_results.csv",
    index=False
)

joblib.dump(
    isolation_model,
    "dataset/isolation_forest_model.pkl"
)

print("Anomaly Results Saved Successfully")
print("dataset/anomaly_results.csv")

print("\nIsolation Forest Model Saved Successfully")
print("dataset/isolation_forest_model.pkl")

Anomaly Results Saved Successfully
dataset/anomaly_results.csv

Isolation Forest Model Saved Successfully
dataset/isolation_forest_model.pkl


In [61]:
# ==========================================
# RISK SCORING
# ==========================================

import pandas as pd
import numpy as np

print("=" * 60)
print("INSIDER RISK SCORING")
print("=" * 60)

# Load anomaly results
risk_data = pd.read_csv("dataset/anomaly_results.csv")

# Anomaly score
score = risk_data["anomaly_score"]

# Normalize score between 0 and 100
min_score = score.min()
max_score = score.max()

if max_score != min_score:
    risk_data["risk_score"] = (
        (score - min_score) /
        (max_score - min_score)
    ) * 100
else:
    risk_data["risk_score"] = 0

# Risk level
def assign_risk_level(score):
    if score >= 80:
        return "Critical Risk"
    elif score >= 60:
        return "High Risk"
    elif score >= 40:
        return "Medium Risk"
    else:
        return "Low Risk"

risk_data["risk_level"] = risk_data["risk_score"].apply(
    assign_risk_level
)

print("\nRisk Score Range")
print("Minimum :", round(risk_data["risk_score"].min(), 2))
print("Maximum :", round(risk_data["risk_score"].max(), 2))

print("\nRisk Level Summary")
print(risk_data["risk_level"].value_counts())

# Save
risk_data.to_csv(
    "dataset/risk_scores.csv",
    index=False
)

print("\nRisk Scores Saved Successfully")
print("dataset/risk_scores.csv")

INSIDER RISK SCORING

Risk Score Range
Minimum : 0.0
Maximum : 100.0

Risk Level Summary
risk_level
Low Risk         43279
Medium Risk       2138
High Risk          524
Critical Risk       54
Name: count, dtype: int64

Risk Scores Saved Successfully
dataset/risk_scores.csv


In [62]:
# ==========================================
# SAMPLE USER RISK SCORE
# ==========================================

sample_user = risk_data.iloc[0]

print("=" * 60)
print("USER RISK RESULT")
print("=" * 60)

print("User       :", sample_user["user"])
print("Risk Score :", round(sample_user["risk_score"], 2))
print("Risk Level :", sample_user["risk_level"])

USER RISK RESULT
User       : AAE0190
Risk Score : 53.82
Risk Level : Medium Risk


In [63]:
# ==========================================
# TOP RISK FACTORS
# ==========================================

sample = risk_data.iloc[0]

risk_features = [
    "logon_count_deviation",
    "off_hours_logons_deviation",
    "distinct_pcs_deviation",
    "usb_connects_deviation",
    "off_hours_usb_deviation",
    "files_copied_to_usb_deviation",
    "sensitive_files_to_usb_deviation",
    "total_emails_sent_deviation",
    "external_emails_sent_deviation",
    "total_attachments_deviation",
    "total_email_size_deviation",
    "http_requests_deviation",
    "cloud_job_visits_deviation"
]

available_features = [
    f for f in risk_features
    if f in sample.index
]

factors = []

for feature in available_features:
    factors.append(
        (feature, abs(float(sample[feature])))
    )

factors = sorted(
    factors,
    key=lambda x: x[1],
    reverse=True
)

print("=" * 60)
print("TOP RISK FACTORS")
print("=" * 60)

for feature, value in factors[:5]:
    print(f"- {feature} : {value:.2f}")

TOP RISK FACTORS
- total_email_size_deviation : 308596.24
- http_requests_deviation : 139.24
- total_emails_sent_deviation : 9.63
- external_emails_sent_deviation : 9.63
- cloud_job_visits_deviation : 3.89


In [64]:
# ==========================================
# EXPLAIN PREDICTION - FINAL OUTPUT
# ==========================================

# Get the first sample
sample = risk_data.iloc[0]

# Calculate total risk score from top deviations
risk_values = [value for feature, value in factors]

if risk_values:
    top_risk_score = sum(risk_values[:5])
else:
    top_risk_score = 0

# Determine risk level
if top_risk_score >= 100:
    risk_level = "HIGH"
elif top_risk_score >= 20:
    risk_level = "MEDIUM"
else:
    risk_level = "LOW"

# Prediction based on risk level
if risk_level == "HIGH":
    prediction = "Potential Insider Threat"
elif risk_level == "MEDIUM":
    prediction = "Suspicious User"
else:
    prediction = "Normal User"


# ==========================================
# FINAL EXPLANATION
# ==========================================

print("=" * 60)
print("EXPLAIN PREDICTION")
print("=" * 60)

print(f"Risk Level : {risk_level}")
print(f"Prediction : {prediction}")

print("\nImportant Factors:")
print("-" * 60)

for feature, value in factors[:5]:
    print(f"{feature} : {value:.2f}")

print("=" * 60)

EXPLAIN PREDICTION
Risk Level : HIGH
Prediction : Potential Insider Threat

Important Factors:
------------------------------------------------------------
total_email_size_deviation : 308596.24
http_requests_deviation : 139.24
total_emails_sent_deviation : 9.63
external_emails_sent_deviation : 9.63
cloud_job_visits_deviation : 3.89


In [67]:
# ==========================================
# FIX ROW COUNT MISMATCH & SAVE ARTIFACTS
# ==========================================
import os
import pickle
import joblib
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from sklearn.ensemble import IsolationForest

# 1. Re-build X from daily_user_features to ensure all 45,995 rows are included
drop_cols = ["user", "day", "date", "activity_date", "Label"]
X_all = daily_user_features.drop(
    columns=[c for c in drop_cols if c in daily_user_features.columns],
    errors="ignore"
).select_dtypes(include=["number"]).fillna(0)

# 2. Scale features for all 45,995 rows
scaler_full = MinMaxScaler()
X_full_scaled = pd.DataFrame(
    scaler_full.fit_transform(X_all),
    columns=X_all.columns
)

# 3. Train Isolation Forest on full dataset
isolation_model = IsolationForest(contamination=0.05, random_state=42)
isolation_model.fit(X_full_scaled)

# 4. Calculate Anomaly Score for all rows matching deviation_data length
deviation_data["anomaly_score"] = -isolation_model.decision_function(X_full_scaled)

# Create output directories
os.makedirs("dataset", exist_ok=True)
os.makedirs("ml_model", exist_ok=True)

# 5. Save Anomaly Results & Models
deviation_data.to_csv("dataset/anomaly_results.csv", index=False)
joblib.dump(isolation_model, "dataset/isolation_forest_model.pkl")

# 6. Save Artifacts for FastAPI Backend
feature_columns_list = list(X_all.columns)
feature_means_dict = daily_user_features[feature_columns_list].mean().to_dict()

pickle.dump(gb, open("ml_model/gb.pkl", "wb"))
pickle.dump(scaler, open("ml_model/scaler.pkl", "wb"))
pickle.dump(feature_means_dict, open("ml_model/feature_means.pkl", "wb"))
pickle.dump(feature_columns_list, open("ml_model/feature_columns.pkl", "wb"))

print("✅ Success! Length mismatch resolved and all model files saved successfully.")

✅ Success! Length mismatch resolved and all model files saved successfully.
